# notebook 08 — `st` 側からは減衰相関は出ない（(2) の前半）

notebook 07 の no-go は「ベル与件 `cos2θ` を保つ限り空間はコンパクト。非コンパクト化には
追加原理が要る」だった。引き継ぎ書 §2-2 の問いは、その追加原理（非コンパクトを生む**減衰
相関**）が、実は `cos2θ` ＋ `st`／作用 の **帰結** として出るのではないか——出れば追加原理は
不要、出なければ no-go が真の必然に格上げ——というものだった。

この notebook は二段検証の **前半（`st` 側）** を担う:

> **`M(f)=st(f/ε^{ord(f)})`（引き継ぎ書 §1-2）のどの使い方かで、`cos2θ` から減衰相関を
> 出せるか。**

これまで私は `st` を「域外対の除外」「位相 `e^{i2θ}` への補完」までしか使っていなかった。
ここでは `st` が減衰を生み得る経路を **網羅的に** 挙げ、全て試す。結論を先取りすると **全て
失敗**だが、失敗の理由が経路ごとに違うので、それを正直に記録する（後半の作用側＝位相転移の
検証は次段）。


## 0. セットアップと `st` 側の経路一覧

`st` の核心は「無限小オーダー `ε^{ord}` で割ってから標準部分を取る」こと。これが**相関の
減衰**を生むには、ある量が分離 `|i−j|` とともに高次の `ε` オーダーに落ちる必要がある。
論理的な経路:

- **S1**: 相関の次数 `ord` が分離とともに増えるか（増えれば実効相関が指数的に切れる）。
- **S2**: 合成・連鎖測定で `ord` が経路長に比例して累積するか。
- **S3**: `ord` 付きの距離再スケールが、連続極限で幾何を開くか。
- **S4**: NSA の `ε²>0` による二次の寄与が、新しい減衰構造を生むか。


In [1]:
import numpy as np
from numpy.linalg import eigh
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def mds(D):
    n=D.shape[0]; J=np.eye(n)-np.ones((n,n))/n
    B=-0.5*J@(D**2)@J; ev,V=eigh(0.5*(B+B.T))
    i=np.argsort(ev)[::-1]; return ev[i], V[:,i]

def signature(ev):
    pos=ev[ev>1e-6]
    if len(pos)==0: return (0,0)
    return (int(np.sum(np.abs(pos-pos[0])<1e-3*pos[0])), len(pos))


## S1. 次数 `ord` は分離とともに増えるか

無限小格子 `θ_i = i·ε` を採る（引き継ぎ書 §1-3 の超有限分割）。標準的な分離 `s=(i−j)ε` が
appreciable であるには `|i−j|~1/ε`（unlimited）が要る。**limited な分離では `(i−j)ε` は無限小**で、
`cos2((i−j)ε)/6 → 1/6`。つまり limited 分離の設定はすべて標準解像度で **一致** し、`st` 幾何は
unlimited 分離だけを見る——そこでは相関は標準変数 `s` の `cos(2s)/6`、すなわち**同じ周期相関**。


In [2]:
eps=1e-3
seps=np.array([0,1,2,10,50])
print("C for limited separations on the eps-lattice (theta_i = i*eps):")
print("  sep =", seps)
print("  C   =", np.round(np.cos(2*seps*eps)/6, 8), " -> all ~ 1/6")
print("\n=> ord does not grow; st recovers cos(2s)/6 in the standard variable s.")
print("   No decay. S1 FAILS (geometry stays the periodic circle).")


C for limited separations on the eps-lattice (theta_i = i*eps):
  sep = [ 0  1  2 10 50]
  C   = [0.1667 0.1667 0.1667 0.1666 0.1658]  -> all ~ 1/6

=> ord does not grow; st recovers cos(2s)/6 in the standard variable s.
   No decay. S1 FAILS (geometry stays the periodic circle).


In [3]:
# S1': the fluctuation f = C - st(C). Its eps-order vs separation.
print("S1': eps-order of the fluctuation C - 1/6 vs separation")
for sep in [1,2,5,10,50]:
    f=np.cos(2*sep*eps)/6 - 1/6
    print(f"  sep={sep:3d}: C-1/6 = {f:.3e}, eps-order ~ {np.log(abs(f))/np.log(eps):.2f}")
print("  -> leading fluctuation is O(eps^2) for all sep; ord does NOT grow. S1' FAILS.")


S1': eps-order of the fluctuation C - 1/6 vs separation
  sep=  1: C-1/6 = -3.333e-07, eps-order ~ 2.16
  sep=  2: C-1/6 = -1.333e-06, eps-order ~ 1.96
  sep=  5: C-1/6 = -8.333e-06, eps-order ~ 1.69
  sep= 10: C-1/6 = -3.333e-05, eps-order ~ 1.49
  sep= 50: C-1/6 = -8.326e-04, eps-order ~ 1.03
  -> leading fluctuation is O(eps^2) for all sep; ord does NOT grow. S1' FAILS.


## S2. 合成・連鎖測定で `ord` は累積するか

合成測定なら `ord` は加法的だが、ベル与件は **二点相関** を `cos2(θ_i−θ_j)/6` に固定する。
連鎖の積 `∏cos2(step)` が与件 `cos2(総差)` に一致するには、距離 `−log|cos2|` が連鎖に沿って
**加法的**でなければならない。`cos2` がこれを満たすか確認する。


In [4]:
step=0.2
print("Is -log|cos2(.)| additive along a chain? (required for a datum-consistent composite)")
for nlinks in [2,3,5]:
    lhs=-np.log(abs(np.cos(2*nlinks*step)))
    rhs=nlinks*(-np.log(abs(np.cos(2*step))))
    print(f"  nlinks={nlinks}: -log|cos2(total)|={lhs:.4f}  sum over links={rhs:.4f}  "
          f"{'MATCH' if abs(lhs-rhs)<1e-9 else 'DIFFER'}")
print("\n=> DIFFER. cos2 distance is non-additive; a chain product != the datum.")
print("   S2 contradicts the two-point Bell datum (confirms notebook 07 L3). FAILS.")


Is -log|cos2(.)| additive along a chain? (required for a datum-consistent composite)
  nlinks=2: -log|cos2(total)|=0.3614  sum over links=0.1645  DIFFER
  nlinks=3: -log|cos2(total)|=1.0151  sum over links=0.2467  DIFFER
  nlinks=5: -log|cos2(total)|=0.8767  sum over links=0.4111  DIFFER

=> DIFFER. cos2 distance is non-additive; a chain product != the datum.
   S2 contradicts the two-point Bell datum (confirms notebook 07 L3). FAILS.


## S3. `ord` 付きの距離再スケールは幾何を開くか

零点 `s_0=π/4` 近傍で `|C|~|s−s_0|/3`、距離 `d=−log|C|` は unlimited。`st(d)` は未定義だが、
`st(d/\log(1/ε))` のような **再スケール**は有限の「接近の次数」を与え、特異点までの距離を測る
**非有界座標** `u=−log(π/4−s)` を作れる。だがこれは与件の二点距離か?


In [5]:
N=200
s=np.linspace(0.01, np.pi/4-1e-6, N)
u=-np.log(np.pi/4 - s)                      # rescaled 'order' coordinate
print(f"rescaled coordinate u range: [{u.min():.2f}, {u.max():.2f}] (UNBOUNDED near s0)")

# (i) MDS of the imposed rescaled distance |u_i - u_j|:
ev_u,V_u=mds(np.abs(u[:,None]-u[None,:]))
print(f"(i) MDS of |u_i-u_j|: signature {signature(ev_u)}, "
      f"coord~u corr={np.corrcoef(V_u[:,0],u)[0,1]:.3f} (a non-compact line)")

# (ii) But the TRUE datum distance between settings s_i, s_j is from cos2(s_i - s_j):
Cd=np.cos(2*(s[:,None]-s[None,:]))/6
with np.errstate(divide='ignore'): Dd=-np.log(np.abs(Cd))
np.fill_diagonal(Dd,0.0); Dd[~np.isfinite(Dd)]=Dd[np.isfinite(Dd)].max()
ev_d,_=mds(Dd)
print(f"(ii) MDS of the TRUE datum distance on [0,pi/4): signature {signature(ev_d)} "
      f"(a compact arc)")
print("\n=> u measures distance to the SINGULARITY, not the pairwise datum correlation.")
print("   Using u as distance is an EXTRA prescription, not a consequence of cos2 + st.")
print("   Within the datum, st-rescaling does NOT yield non-compactness. S3 FAILS.")


rescaled coordinate u range: [0.25, 13.82] (UNBOUNDED near s0)
(i) MDS of |u_i-u_j|: signature (1, 1), coord~u corr=-1.000 (a non-compact line)
(ii) MDS of the TRUE datum distance on [0,pi/4): signature (1, 197) (a compact arc)

=> u measures distance to the SINGULARITY, not the pairwise datum correlation.
   Using u as distance is an EXTRA prescription, not a consequence of cos2 + st.
   Within the datum, st-rescaling does NOT yield non-compactness. S3 FAILS.


## S4. NSA の `ε²>0` による二次の階層

零点で一次（`sin2θ`）が現れ（notebook 06 の `e^{i2θ}` 補完）、さらに二次 `ε²` が…という階層。
だが `st(f/ε^{ord})` は leading な appreciable 項だけを残す。二次以降は `ε` 抑制され `st` が
落とすので、標準部分の像は `{cos2θ, sin2θ}`＝円のまま。


In [6]:
th=np.linspace(0,2*np.pi,100,endpoint=False)
# graded embedding: O(1) phase plus eps-suppressed higher orders
emb=np.stack([np.cos(2*th), np.sin(2*th),
              1e-3*np.cos(2*th), 1e-3*np.sin(2*th)], axis=1)
ev=np.sort(np.linalg.eigvalsh(emb@emb.T))[::-1]
print("graded embedding (incl. eps^2 terms) eigenvalues:", np.round(ev[:5],4))
print("rank of the appreciable (O(1)) part:", int((ev>1e-2).sum()), "-> 2 (circle).")
print("\n=> eps^2 terms are infinitesimal; st drops them. S4 adds nothing standard. FAILS.")


graded embedding (incl. eps^2 terms) eigenvalues: [50. 50.  0.  0.  0.]
rank of the appreciable (O(1)) part: 2 -> 2 (circle).

=> eps^2 terms are infinitesimal; st drops them. S4 adds nothing standard. FAILS.


## まとめ：`st` 側からは減衰相関は出ない

### 全経路の結果

| 経路 | 内容 | 判定 | 理由 |
|---|---|---|---|
| S1 | `ord` が分離で増えるか | **失敗** | `ε`格子で `ord` 増えず、`st` は周期 `cos(2s)/6` を再現 |
| S1' | 揺らぎ `C−st(C)` の `ord` | **失敗** | `ord~2` 一定、分離に依らず減衰しない |
| S2 | 合成測定の `ord` 累積 | **失敗** | `cos2` 距離が非加法的 → 与件違反（nb07 L3 再確認） |
| S3 | `ord` 付き距離の再スケール | **失敗** | 非有界座標 `u` は特異点までの距離で、与件の二点距離でない |
| S4 | `ε²>0` の二次階層 | **失敗** | `ε²` 項は無限小、`st` が落とす。標準部分は円 |

### 共通の根因（誠実版）
`st` は **無限小オーダーを落として標準部分を取る** 操作であり、`cos2θ` の周期的・有界・
ランク2 の構造を「円上の操作」として **保存** する。`st` は点を除外・補完するが、**与件が
定める二点相関の関数形そのものは変えられない**。減衰相関は二点相関の関数形の変更（道I）で
あり、`st` の守備範囲の外。

### この段の結論
**減衰相関は `st` 側からは出ない**（数値的に強く支持）。したがって §2-2 の問い「追加原理は
不要か」に対する `st` 側の答えは **否定的**：`st` だけでは no-go を回避できない。

ただし **これは (2) の前半に過ぎない**。残る可能性は **作用 `S=−βΣe^{−d}` の実効相関**で、
統計力学では素の結合が有界でも実効二点相関が指数減衰する（相関長が有限）ことは普通に
起きる。そこに **位相転移**の可能性がある。これを後半（次段）で検証する。

### 規律の自己点検（引き継ぎ書 §6）
- `st` の使い方を4経路で網羅し、全て失敗として正直に記録した。✅
- S3 で「非有界座標が作れた」ことに飛びつかず、それが与件の距離でないと明示した。✅
- 「`st` で減衰が出ない」を断定しすぎず、作用側という残る可能性を明記した。✅
- no-go 回避の可否を、`st` 側に限った否定として限定的に述べた。✅

### 次への申し送り（後半＝作用側）
1. **転送作用素のスペクトルギャップ**：作用 `S=−βΣe^{−d}` の転送核 `T(θ,θ')=exp(β|C|...)`
   の第二固有値が、実効二点相関の **相関長** を決める。ギャップが開けば実効相関は指数減衰
   ＝非コンパクト方向の種。
2. **位相転移**：`β`（または `N`）を変えたとき、相関長が発散する臨界点があるか。臨界点を
   またいでコンパクト↔非コンパクトが切り替わるなら、それが「追加原理」の正体（自発的）か。
3. 作用側でも出なければ、no-go は `st`・作用の両面で確定し、**追加原理の存在が必然**となる。
